<a href="https://colab.research.google.com/github/keyvanmahmoudi/Data_Science_Course/blob/main/solutions/Session03_data_cleaning_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۳: پاک‌سازی داده با Pandas

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session03_data_cleaning.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

کتابخانه‌ها و داده‌هایی که پاسخ‌ها به آن‌ها نیاز دارند:

</div>

In [1]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/keyvanmahmoudi/Data_Science_Course/main/data/"
raw = pd.read_csv(DATA_URL + "orders_raw.csv")
products = pd.read_csv(DATA_URL + "products.csv")
orders = pd.read_csv(DATA_URL + "orders_clean.csv", parse_dates=["order_date"])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

در `raw`، چند سفارش مبلغی دارند که بر قیمت واحد کالا (از فایل `products.csv`) بخش‌پذیر نیست؟ این یعنی چه؟ (راهنما: تخفیف)

</div>

In [2]:
df = raw.drop_duplicates().copy()
df["total_price"] = df["total_price"].astype(str).str.replace(",", "", regex=False).astype("int64")
df = df[df["quantity"] > 0].merge(products[["product_id", "unit_price"]], on="product_id")

full_price = df["unit_price"] * df["quantity"]
not_divisible = df["total_price"] % df["unit_price"] != 0
print("orders whose total is not a multiple of the unit price:", not_divisible.sum())
discount = (1 - df["total_price"] / full_price).round(2)
discount[not_divisible].value_counts().sort_index()

orders whose total is not a multiple of the unit price: 586


0.05    205
0.10    191
0.20    190
Name: count, dtype: int64

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

این سفارش‌ها **تخفیف** گرفته‌اند. نسبت مبلغ پرداختی به قیمت کامل نشان می‌دهد تخفیف‌ها ۵، ۱۰ و ۲۰ درصد بوده‌اند (اختلاف‌های کوچک به‌خاطر گرد شدن مبلغ تا هزار تومان است). سفارش‌هایی که تخفیف نگرفته‌اند یا مبلغشان پس از تخفیف اتفاقاً مضرب قیمت واحد شده، در این شمارش نیستند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

ستونی به نام `is_weekend` بسازید که برای سفارش‌های پنجشنبه و جمعه `True` باشد.

</div>

In [3]:
orders["weekday"] = orders["order_date"].dt.day_name()
orders["is_weekend"] = orders["weekday"].isin(["Thursday", "Friday"])
orders["is_weekend"].value_counts()

is_weekend
False    890
True     306
Name: count, dtype: int64

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

به‌جای میانه، `delivery_days` گمشده را با میانهٔ **همان شهر** پر کنید. (راهنما: `groupby(...).transform("median")`)

</div>

In [4]:
df = raw.drop_duplicates().copy()
df["city"] = df["city"].str.strip().str.title().replace({"Esfahan": "Isfahan", "Mashad": "Mashhad"})
df["delivery_days"] = df["delivery_days"].mask(df["delivery_days"] > 60)
print("missing before:", df["delivery_days"].isna().sum())

city_median = df.groupby("city")["delivery_days"].transform("median")
df["delivery_days"] = df["delivery_days"].fillna(city_median).fillna(df["delivery_days"].median())
print("missing after :", df["delivery_days"].isna().sum())
df.groupby("city")["delivery_days"].median()

missing before: 3
missing after : 0


city
Amol       4.0
Isfahan    4.0
Mashhad    4.0
Rasht      4.0
Sari       4.0
Shiraz     4.0
Tabriz     4.0
Tehran     4.0
Name: delivery_days, dtype: float64

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

با روش IQR، سفارش‌های پرت از نظر `total_price` را پیدا کنید. آیا به نظرتان غلط هستند یا واقعی؟

</div>

In [5]:
q1, q3 = orders["total_price"].quantile([0.25, 0.75])
iqr = q3 - q1
high = q3 + 1.5 * iqr
outliers = orders[orders["total_price"] > high].merge(products, on="product_id")
print(f"upper fence = {high:,.0f} Toman, outliers = {len(outliers)}")
outliers.groupby("product_name")["quantity"].agg(["count", "mean"]).sort_values("count", ascending=False)

upper fence = 7,160,000 Toman, outliers = 96


,count,mean
product_name,,
Smart Watch,35,2.771429
Bluetooth Speaker,21,3.380952
Rug (small),15,3.533333
Wireless Earbuds,12,3.833333
Pistachio 500g,8,4.000000
Sneakers,5,3.400000


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

تقریباً همهٔ سفارش‌های پرت، خرید چند عدد از کالاهای گران (ساعت هوشمند، اسپیکر، فرش، کفش و ...) هستند. پس این‌ها **واقعی‌اند** و نباید حذف شوند؛ فقط توزیع مبلغ سفارش چوله به راست است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

تابع `clean_orders` را طوری تغییر دهید که گزارشی از تعداد سطرهای حذف‌شده در هر مرحله چاپ کند.

</div>

In [6]:
CITY_FIX = {"Esfahan": "Isfahan", "Mashad": "Mashhad"}

def clean_orders(raw: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    def report(step, before, after):
        if verbose:
            print(f"{step:32s} {before:5d} -> {after:5d} rows  (removed {before - after})")

    n0 = len(raw)
    df = raw.drop_duplicates().copy()
    report("drop exact duplicates", n0, len(df))

    df["total_price"] = df["total_price"].astype(str).str.replace(",", "", regex=False).astype("int64")
    df["order_date"] = pd.to_datetime(df["order_date"], format="%Y-%m-%d %H:%M")
    df["city"] = df["city"].str.strip().str.title().replace(CITY_FIX)
    df["payment_method"] = (df["payment_method"].str.strip().str.lower()
                            .replace({"cod": "cash_on_delivery"}).fillna("unknown"))

    n1 = len(df)
    df = df[df["quantity"] > 0].copy()
    report("drop non-positive quantity", n1, len(df))

    df["rating"] = df["rating"].where(df["rating"].between(1, 5))
    df["delivery_days"] = df["delivery_days"].mask(df["delivery_days"] > 60)
    df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median()).astype(int)
    known_city = df.dropna(subset=["city"]).groupby("customer_id")["city"].agg(lambda s: s.mode().iloc[0])
    df["city"] = df["city"].fillna(df["customer_id"].map(known_city))
    return df.reset_index(drop=True)

clean = clean_orders(raw)
print(clean.shape)

drop exact duplicates             1218 ->  1200 rows  (removed 18)
drop non-positive quantity        1200 ->  1196 rows  (removed 4)
(1196, 10)
